# Projektarbeit: Applied Neural Networks

**Projektname:** Kleiderschrank sortieren

**Autoren:** Daria Lukyanova, Carola Bürgi

**Datum:** 15.06.2026

**Institution:** Ostschweizer Fachhochschule | Systemtechnik BSc | FS 2026

---

## 1. Einleitung

### 1.1 Aufgabenbeschreibung
Ziel dieses Projekts ist die Entwicklung und Implementierung eines Analytical Neural Networks (ANN) zur automatischen Klassifikation von Kleidungsstücken anhand von Bilddaten. Das Modell wird mit Hilfe der Bibliothek PyTorch trainiert und soll verschiedene Kleidungsarten zuverlässig erkennen und in die Kategorien Shorts, Schuhe, Hosen, Kleider und Shirts einteilen. Grundlage bildet ein realer Bilddatensatz aus einer öffentlich verfügbaren Quelle.

### 1.2 Anwendungsbereich
Die automatische Klassifikation von Kleidung ist ein typisches Problem im Bereich der Bildverarbeitung und des maschinellen Lernens. Praktische Anwendungen finden sich insbesondere im E-Commerce, wo Produkte automatisch kategorisiert und effizient verwaltet werden müssen.

Weitere Einsatzbereiche sind beispielsweise visuelle Suchsysteme, Empfehlungssysteme oder automatisierte Sortierprozesse in der Logistik. Auch in der Modeindustrie kann eine solche Technologie zur Analyse von Trends oder zur Unterstützung von Designprozessen eingesetzt werden.


### 1.3 Bedeutung des Themas
Durch dieses Projekt wird ein praktisches Verständnis für den Aufbau, das Training und die Optimierung von neuronalen Netzen entwickelt. Zudem zeigt es exemplarisch, wie theoretische Konzepte aus dem Bereich Machine Learning auf reale Datensätze angewendet werden können.

Die Bedeutung dieses Themas liegt in der zunehmenden Relevanz datengetriebener Systeme und künstlicher Intelligenz in industriellen und kommerziellen Anwendungen. Bildklassifikation mit neuronalen Netzen gehört zu den zentralen Aufgaben moderner KI-Systeme.

## 2. Zielsetzung und Vorgehensweise

### 2.1 Zielsetzung
Ziel dieses Projekts ist die Entwicklung eines leistungsfähigen Klassifikationsmodells für Kleidungsbilder auf Basis eines neuronalen Netzes. Dabei soll das vortrainierte Modell MobileNetV2 im Rahmen von Transfer Learning so angepasst werden, dass es die definierten Kategorien (Shorts, Schuhe, Hosen, Kleider, Shirts) möglichst zuverlässig erkennt.

Ein weiterer Fokus liegt auf der effizienten Nutzung vorhandener Rechenressourcen sowie auf der Reduktion der Trainingszeit, ohne dabei signifikante Einbussen bei der Modellgenauigkeit zu erzielen.

### 2.2 Adressierte Fragen
- Wie gut eignet sich Transfer Learning mit MobileNetV2 für die Klassifikation von Kleidungsbildern?
- Wie beeinflusst die Wahl von Hyperparametern die Performance des Modells?


### 2.3 Vorgehensweise
Zum Abschluss wird das trainierte Modell auf den Testdaten evaluiert und hinsichtlich Genauigkeit sowie Generalisierungsfähigkeit analysiert.

Der bereits gelabelte und strukturierte Datensatz wird direkt in Trainings-, Validierungs- und Testdaten aufgeteilt und für das Modelltraining vorbereitet.

Das Training erfolgt mittels Transfer Learning, wobei die vortrainierten Gewichte als Ausgangspunkt genutzt werden. Dabei werden ausgewählte Hyperparameter gezielt variiert, um die Modellleistung zu optimieren.

Anschliessend wird das vortrainierte Modell MobileNetV2 geladen und für die spezifische Klassifikationsaufgabe angepasst, indem die finale Klassifikationsschicht auf die vorhandenen Kategorien abgestimmt wird.

## 3. Datenladenoption und Preprocessing

### 3.1 Datenbeschreibung
Für dieses Projekt wird das Apparel Images Dataset verwendet, welches eine grosse Anzahl von Bildern unterschiedlicher Kleidungsstücke enthält. Die Bilder sind bereits kategorisiert und in klar definierte Klassen unterteilt, darunter Shorts, Schuhe, Hosen, Kleider und Shirts.

Der Datensatz zeichnet sich durch eine hohe Variabilität aus, da die Kleidungsstücke in unterschiedlichen Perspektiven, Farben und Formen dargestellt sind. Dadurch eignet er sich gut für die Anwendung von Deep Learning, da das Modell robuste visuelle Merkmale lernen muss, um die Klassen zuverlässig zu unterscheiden.

Die Daten liegen in strukturierter Form vor, sodass sie direkt für das Training eines neuronalen Netzes verwendet werden können.

### 3.2 Datenquellen
Der verwendete Datensatz stammt von der Plattform Kaggle und ist öffentlich zugänglich:
https://www.kaggle.com/datasets/trolukovich/apparel-images-dataset/data

#### 3.2.1 Imortierung aller notwendigen Libraries

In [1]:

# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
import kagglehub as kh

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import MobileNet_V2_Weights

# Check device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu
PyTorch Version: 2.11.0+cpu
Torchvision Version: 0.26.0+cpu


#### 3.2.2: Laden und erkunden des Datensatzes

In [ ]:
# Define the dataset path (customize this path to your local dataset location)
dataset_path = r'E:\daria\FS26\ANN\ANN_Project\project_ann\dataset'  # Change this to your dataset path

# Check if dataset exists
if os.path.exists(dataset_path):
    print(f"Dataset found at: {dataset_path}")
    print("\nDirectory structure:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{sub_indent}{file}')
        if len(files) > 3:
            print(f'{sub_indent}... and {len(files) - 3} more files')
else:
    print(f"Dataset not found at {dataset_path}")
    print("Please download the dataset from: https://www.kaggle.com/datasets/trolukovich/apparel-images-dataset")
    print("And update the 'dataset_path' variable with the correct location.")

Pfad zum Datensatz: C:\Users\Carola\.cache\kagglehub\datasets\trolukovich\apparel-images-dataset\versions\6


NameError: name 'clothingDataModule' is not defined

In [ ]:
# Count the number of images and classes in the dataset
def analyze_dataset(dataset_path):
    """
    Analyze the structure of the apparel dataset
    Returns: class_count dictionary with class names and number of images
    """
    class_count = {}
    
    for category in os.listdir(dataset_path):
        category_path = os.path.join(dataset_path, category)
        if os.path.isdir(category_path):
            image_files = [f for f in os.listdir(category_path) 
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
            class_count[category] = len(image_files)
    
    return class_count

# Analyze dataset if it exists
if os.path.exists(dataset_path):
    class_counts = analyze_dataset(dataset_path)
    
    print("Dataset Analysis:")
    print(f"Total number of classes: {len(class_counts)}")
    print(f"\nClass distribution:")
    for class_name, count in sorted(class_counts.items()):
        print(f"  {class_name}: {count} images")
    
    total_images = sum(class_counts.values())
    print(f"\nTotal images: {total_images}")

In [ ]:
# Display sample images from each class
from PIL import Image

def display_sample_images(dataset_path, samples_per_class=2):
    """
    Display sample images from each clothing category
    """
    classes = os.listdir(dataset_path)
    classes = [c for c in classes if os.path.isdir(os.path.join(dataset_path, c))]
    
    num_classes = len(classes)
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(12, 3 * num_classes))
    
    for idx, class_name in enumerate(sorted(classes)):
        class_path = os.path.join(dataset_path, class_name)
        images = [f for f in os.listdir(class_path) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
        
        for sample_idx in range(min(samples_per_class, len(images))):
            ax = axes[idx, sample_idx] if num_classes > 1 else axes[sample_idx]
            img_path = os.path.join(class_path, images[sample_idx])
            
            try:
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f'{class_name}')
                ax.axis('off')
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
    
    plt.tight_layout()
    plt.show()

# Display samples if dataset exists
if os.path.exists(dataset_path):
    print("Displaying sample images from each clothing category:")
    display_sample_images(dataset_path, samples_per_class=2)

## 4. Modellaufbau und Beschreibung der Modell-Architektur

### 4.1 Netzwerk-Architektur
Für die Klassifikationsaufgabe wird das vortrainierte Convolutional Neural Network MobileNetV2 verwendet. Das Modell basiert auf einer tiefen CNN-Architektur, die speziell für effiziente Bildverarbeitung optimiert ist.

MobileNetV2 besteht aus mehreren Faltungsblöcken, die sogenannte Inverted Residual Blocks verwenden. Diese kombinieren Depthwise-Separable Convolutions mit linearen Bottlenecks, wodurch die Anzahl der Parameter und der Rechenaufwand reduziert werden.

Für die vorliegende Aufgabe wird das vortrainierte Modell übernommen und die finale Klassifikationsschicht ersetzt. Die neue Fully-Connected-Schicht wird so angepasst, dass sie die fünf Zielklassen (Shorts, Schuhe, Hosen, Kleider, Shirts) ausgibt.

### 4.2 Begründung der Architekturwahl
MobileNetV2 wurde gewählt, da es ein sehr gutes Verhältnis zwischen Modellgenauigkeit und Rechenaufwand bietet. Dies ist insbesondere relevant, da das Modell auf begrenzten Ressourcen (z. B. lokaler Rechner oder Colab) trainiert wird.

Durch die Verwendung von Transfer Learning können bereits gelernte Features aus dem ImageNet-Datensatz genutzt werden. Dadurch wird die Trainingszeit deutlich reduziert und gleichzeitig die Modellleistung verbessert, insbesondere im Vergleich zu einem von Grund auf trainierten Netzwerk.

### 4.3 Besonderheiten der Architektur
Eine zentrale Besonderheit von MobileNetV2 ist die Verwendung von Inverted Residual Blocks. Im Gegensatz zu klassischen CNNs wird dabei zunächst die Dimension erhöht und anschliessend wieder reduziert, was eine effizientere Feature-Extraktion ermöglicht.

Zusätzlich verwendet das Modell Depthwise-Separable Convolutions, bei denen räumliche und kanalweise Faltung getrennt durchgeführt werden. Dies reduziert den Rechenaufwand erheblich, ohne die Performance stark zu beeinträchtigen.

Ein weiterer Vorteil ist die kompakte Modellstruktur, die eine schnelle Trainings- und Inferenzzeit ermöglicht.

### 4.4 Mögliche Alternativen
Als Alternative zu MobileNetV2 könnten klassische Convolutional Neural Networks verwendet werden, die von Grund auf trainiert werden. Diese sind jedoch in der Regel rechenintensiver und benötigen mehr Trainingsdaten, um vergleichbare Ergebnisse zu erzielen.

Weitere mögliche Architekturen sind beispielsweise ResNet oder EfficientNet. Diese bieten teilweise höhere Genauigkeiten, gehen jedoch oft mit einem höheren Rechenaufwand einher.

### 4.5 Datenvorbereitung für das MobileNetV2 CNN

In [ ]:
# Define image preprocessing parameters and transformations
IMG_HEIGHT = 224  # MobileNetV2 standard input height
IMG_WIDTH = 224   # MobileNetV2 standard input width
BATCH_SIZE = 32   # Number of images to process at once

# Training transformations: includes augmentation for better generalization
# Augmentation: rotation, horizontal flip, color jitter, etc.
train_transforms = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),  # Resize to 224x224
    transforms.RandomRotation(20),                 # Random rotation up to 20 degrees
    transforms.RandomHorizontalFlip(0.5),          # 50% chance of horizontal flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Color variations
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Random translation
    transforms.ToTensor(),                         # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                        std=[0.229, 0.224, 0.225])
])

# Validation transformations: no augmentation, only resizing and normalization
val_transforms = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Data preprocessing configuration:")
print(f"  Image size: {IMG_HEIGHT} x {IMG_WIDTH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training transformations: Rotation, Flip, ColorJitter, Normalize")
print(f"  Validation transformations: Resize, Normalize")

In [ ]:
# Create a custom dataset class that splits data into train/validation
class ApparelDataset(Dataset):
    def __init__(self, dataset_path, transform=None, split='train', split_ratio=0.8):
        """
        Custom dataset for apparel images with train/validation split
        """
        self.dataset_path = dataset_path
        self.transform = transform
        self.split = split
        self.split_ratio = split_ratio
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        
        # Load all images and create class indices
        class_names = sorted([d for d in os.listdir(dataset_path) 
                             if os.path.isdir(os.path.join(dataset_path, d))])
        
        for idx, class_name in enumerate(class_names):
            self.class_to_idx[class_name] = idx
            class_path = os.path.join(dataset_path, class_name)
            
            for img_file in os.listdir(class_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
                    self.images.append(os.path.join(class_path, img_file))
                    self.labels.append(idx)
        
        # Split into train/validation
        split_idx = int(len(self.images) * split_ratio)
        if split == 'train':
            self.images = self.images[:split_idx]
            self.labels = self.labels[:split_idx]
        else:  # validation
            self.images = self.images[split_idx:]
            self.labels = self.labels[split_idx:]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 224, 224), label

# Load datasets and create DataLoaders
if os.path.exists(dataset_path):
    train_dataset = ApparelDataset(dataset_path, transform=train_transforms, split='train')
    val_dataset = ApparelDataset(dataset_path, transform=val_transforms, split='val')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Store class information
    class_to_idx = train_dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    num_classes = len(class_to_idx)
    
    print(f"Successfully loaded data!")
    print(f"Number of classes: {num_classes}")
    print(f"Classes: {list(class_to_idx.keys())}")
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
else:
    print("Dataset path not found. Please update dataset_path variable.")

### 4.6 Transfer Learning Model bilden

In [ ]:
# Load pretrained MobileNetV2 model
base_model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

print(f"Base Model: MobileNetV2")
print(f"Model loaded with ImageNet pre-trained weights")
print(f"Model structure:")
print(base_model)

In [ ]:
# Freeze base model layers
# This prevents the pre-trained weights from being updated during training
for param in base_model.parameters():
    param.requires_grad = False

# Count trainable parameters
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in base_model.parameters())

print(f"\nBase model layer freeze status:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  All layers frozen: {trainable_params == 0}")

## 5. Hyperparameter-Optimierung

### 5.1 Modell-Hyperparameter


### 5.2 Algorithmus-Hyperparameter


### 5.3 Optimierungsstrategie


## 6. Validierung

### 6.1 Validierungsstrategie

### 6.2 Performance-Metriken


### 6.3 Ergebnisse und Diskussion


## 7. Schlussfolgerung und Ausblick

### 7.1 Zusammenfassung der Ergebnisse

### 7.2 Erreichte Ziele

### 7.3 Herausforderungen und Limitierungen

### 7.4 Weiterführende Untersuchungen


### 7.5 Methodische Alternativen


## 8. Referenzen

### Quellen
- Vorlesungsunterlagen Modul ANN [Christoph Würsch]

---

## Eigenständigkeitserklärung

Hiermit bestätigen bestätigen wir, dass wir die vorliegende Arbeit selbständig verfasst und keine anderen als die angegebenen Hilfsmittel benutzt haben. Die Stellen der Arbeit, die dem Wortlaut oder dem Sinn nach anderen Werken (dazu zählen auch Internetquellen) entnommen sind, wurden unter Angabe der Quelle kenntlich gemacht.

**Datum:** 14.06.2026

**Unterschrift(en):** 

Daria Lukyanova

Carola Bürgi